<a href="https://colab.research.google.com/github/marcelonicolays/tcc_compras_diretas/blob/main/SQL_TCC_compras_diretas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCP - Universidade Católica de Petrópolis
## Projeto de Conclusão de Curso
Marcelo Nicolay Santos

12510824

Dados extraídos no notebook ETL_TCC_compras_diretas

* Instalar e importar bibliotecas

In [1]:
!pip install sqlalchemy psycopg2-binary pandas pyarrow

In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files
from sqlalchemy import create_engine, text
from getpass import getpass
from sqlalchemy.engine import URL

###  Importar arquivo df_limpo.csv

In [3]:
# Define a pasta na raiz da sessão do Colab (/content)
pasta_dados = Path("/content")
arquivo_parquet = pasta_dados / "df_limpo.parquet"

df_limpo = None

# 1. Verifica se já existe um arquivo salvo
if arquivo_parquet.exists():
    try:
        print("Arquivo CSV encontrado. Carregando dados locais...")
        df_limpo = pd.read_parquet(arquivo_parquet)
        print("Arquivo carregado com sucesso.")

    except Exception as e:
        print(f"Erro ao ler o arquivo local: {e}")
        print("Realizar download do arquivo")

# Fazer upload do arquivo

if df_limpo is None:
  try:
    print("Por favor, faça o upload do arquivo clicando no botão abaixo:")
    uploaded = files.upload()
    nome_enviado = list(uploaded.keys())[0]
    if nome_enviado != 'df_limpo.parquet':
        os.rename(nome_enviado, arquivo_parquet)
        print(f"🔄 Arquivo renomeado de '{nome_enviado}' para 'df_limpo.parquet'.")
  except Exception as e:
    print(f"Falha ao acessar ou processar o arquvo: {e}")
    raise RuntimeError(
      "Não foi possível obter os dados."
    ) from e

display(df_limpo.head())
df_limpo.info()

Arquivo CSV encontrado. Carregando dados locais...
Arquivo carregado com sucesso.


,Processo,ValorProcesso,Objeto,Afastamento,AnoProcesso,DataAprovacao,EnquadramentoLegal,Unidade,FornecedorVencedor,Item,Quantidade,UnidadeMedida,ValorUnitario
0,**SEI-120001/009348/2022,10994.0,CONTRATAÇÃO DE PESSOA JURÍDICA ESPECIALIADA EM...,Dispensa - Pequenas Compras,2023,2023-07-05,"De acordo com o artigo24, inciso 2º da Lei Fed...",SEPLAG - Secretaria de Estado de Planejamento ...,MAR E FIRE EQUIP. E PROT. CONTRA INCÊNDIO LTDA-ME,MANUTENCAO EM EXTINTORES DE INCENDIO - DESCRI...,1.0,SERVICO,10994.0
1,020002/000001/2024,936.0,PRESTAÇÃO DE SERVIÇOS DE CARÁTER CONTINUADO CO...,Inexigibilidade - 14.133/2021,2024,2024-02-08,"Lei nº 14.133/2021, Art. 74º, I",EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...,ÁGUAS DAS AGULHAS NEGRAS S/A,SERVICO DE FORNECIMENTO DE AGUA - DESCRIÇÃO: C...,12.0,SERVICO,78.0
2,020002/000002/2024,3696.0,GERAMOS O PROCESSO ADMINISTRATIVO PARA O EMPEN...,Inexigibilidade - 14.133/2021,2024,2024-02-08,"Lei nº 14.133/2021, Art. 74º, I",EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...,AUTARQUIA MUNICIPAL DE ÁGUA E ESGOTO DE CACHOE...,SERVICO DE FORNECIMENTO DE AGUA - DESCRIÇÃO: C...,12.0,SERVICO,308.0
3,020002/000003/2024,5040.0,GERAMOS O PROCESSO ADMINISTRATIVO PARA O EMPEN...,Inexigibilidade - 14.133/2021,2024,2024-02-08,"Lei nº 14.133/2021, Art. 74º, I",EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...,CONCESSIONÁRIA ÁGUAS DE JUTURNAÍBA S/A,SERVICO DE FORNECIMENTO DE AGUA - DESCRIÇÃO: C...,12.0,SERVICO,420.0
4,020002/000004/2024,3120.0,GERAMOS O PROCESSO ADMINISTRATIVO PARA O EMPEN...,Inexigibilidade - 14.133/2021,2024,2024-02-08,"Lei nº 14.133/2021, Art. 74º, I",EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...,AGUAS DE NITEROI S/A,SERVICO DE FORNECIMENTO DE AGUA - DESCRIÇÃO: C...,12.0,SERVICO,260.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20642 entries, 0 to 20641
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Processo            20642 non-null  object        
 1   ValorProcesso       20642 non-null  float64       
 2   Objeto              20642 non-null  object        
 3   Afastamento         20642 non-null  object        
 4   AnoProcesso         20642 non-null  Int64         
 5   DataAprovacao       20642 non-null  datetime64[ns]
 6   EnquadramentoLegal  20642 non-null  object        
 7   Unidade             20642 non-null  object        
 8   FornecedorVencedor  20642 non-null  object        
 9   Item                20642 non-null  object        
 10  Quantidade          20642 non-null  float64       
 11  UnidadeMedida       20642 non-null  object        
 12  ValorUnitario       20642 non-null  float64       
dtypes: Int64(1), datetime64[ns](1), float64(3), ob

## Conexão com Supabase

In [4]:
# insere dados do banco
host = input('Host do Session pooler: ').strip()
usuario = input('Usuário do banco: ').strip()
senha = getpass('Senha do banco: ')

# Monta URL
url = URL.create(
    drivername='postgresql+psycopg2',
    username=usuario,
    password=senha,
    host=host,
    port=5432,
    database='postgres'
)

# Cria Engine
engine = create_engine(
    url,
    connect_args={
        'sslmode': 'require',
        'connect_timeout': 15
    },
    pool_pre_ping=True,
    pool_size=1,
    max_overflow=0
)

# Testa a conexão
try:
    with engine.connect() as conn:
        resultado = conn.execute(
            text('SELECT current_database(), current_schema()')
        ).fetchone()

    print('Conexão realizada com sucesso!')
    print(f'Banco: {resultado[0]}')
    print(f'Schema: {resultado[1]}')
except Exception as e:
    print(f'Erro na conexão: {e}')

Host do Session pooler: aws-0-us-east-1.pooler.supabase.com
Usuário do banco: postgres.ydrjsitssroilllyyoqf
Senha do banco: ··········
Conexão realizada com sucesso!
Banco: postgres
Schema: public


* Limpeza do Banco

In [5]:
# Listar todas as tabelas para que tudo seja limpo.
tabelas = [
    'itens_processo', 'processo_compra_direta', 'processo', 'item',
    'cnae_parte', 'socio', 'parte_pf', 'parte_pj', 'parte',
    'orgao_estadual', 'orgao_municipal', 'orgao', 'cep',
    'municipio', 'uf', 'afastamento', 'un_medida',
    'categoria_item', 'natureza_juridica', 'poder', 'esfera', 'tipo_contratacao', 'cnae'
]

with engine.begin() as conn:
    # O CASCADE remove os registros mesmo que existam FKs apontando para eles
    conn.execute(text(f"TRUNCATE TABLE {', '.join(tabelas)} RESTART IDENTITY CASCADE;"))

print("Banco limpo e pronto para receber os dados!")

Banco limpo e pronto para receber os dados!


## Preparação e Dimensões Básicas

In [6]:
# 1. Padronização de Strings (remove espaços em branco extras no início e fim)
colunas_texto = ['Afastamento', 'Unidade', 'FornecedorVencedor', 'Item', 'UnidadeMedida', 'Objeto', 'EnquadramentoLegal']
for col in colunas_texto:
    df_limpo[col] = df_limpo[col].astype(str).str.strip()

# 2. Função auxiliar para carregar dimensões e mapear IDs
def carregar_dimensao(df, col_df, tabela, col_db, id_col_db):
    valores_unicos = df[col_df].dropna().unique()
    df_novos = pd.DataFrame({col_db: valores_unicos})

    df_existente = pd.read_sql(f"SELECT {col_db} FROM {tabela}", engine)
    df_inserir = df_novos[~df_novos[col_db].isin(df_existente[col_db])]

    if not df_inserir.empty:
        df_inserir.to_sql(tabela, engine, if_exists='append', index=False)

    df_ref = pd.read_sql(f"SELECT {id_col_db}, {col_db} FROM {tabela}", engine)

    return df.merge(df_ref, left_on=col_df, right_on=col_db, how='left')

print("Carregando dimensões fixas (Placeholders)...")

# UF (Assume-se RJ para todo o escopo)
if pd.read_sql("SELECT * FROM uf WHERE uf = 'RJ'", engine).empty:
    pd.DataFrame({'uf': ['RJ'], 'nome_estado': ['Rio de Janeiro']}).to_sql('uf', engine, if_exists='append', index=False)
uf_id = pd.read_sql("SELECT iduf FROM uf WHERE uf = 'RJ'", engine)['iduf'].values[0]

# Poder, Esfera e Tipo de Contratação
if pd.read_sql("SELECT * FROM poder", engine).empty:
    pd.DataFrame({'poder': ['Executivo']}).to_sql('poder', engine, if_exists='append', index=False)
if pd.read_sql("SELECT * FROM esfera", engine).empty:
    pd.DataFrame({'esfera': ['Estadual']}).to_sql('esfera', engine, if_exists='append', index=False)
if pd.read_sql("SELECT * FROM tipo_contratacao WHERE descricao = 'Compra Direta'", engine).empty:
    pd.DataFrame({'descricao': ['Compra Direta']}).to_sql('tipo_contratacao', engine, if_exists='append', index=False)

# Natureza Jurídica e Categoria de Item (Não vêm na base de dados, usamos placeholders)
if pd.read_sql("SELECT * FROM natureza_juridica", engine).empty:
    pd.DataFrame({'natureza': ['Não Informada']}).to_sql('natureza_juridica', engine, if_exists='append', index=False)
natjur_id = pd.read_sql("SELECT idnatureza_juridica FROM natureza_juridica LIMIT 1", engine)['idnatureza_juridica'].values[0]

if pd.read_sql("SELECT * FROM categoria_item", engine).empty:
    pd.DataFrame({'descricao': ['Outros']}).to_sql('categoria_item', engine, if_exists='append', index=False)
catitem_id = pd.read_sql("SELECT idcategoria_item FROM categoria_item LIMIT 1", engine)['idcategoria_item'].values[0]

# 3. Carrega Afastamento e Unidade de Medida a partir do DF
print("Carregando Afastamentos e Unidades de Medida...")
df_limpo = carregar_dimensao(df_limpo, 'Afastamento', 'afastamento', 'descricao_afastamento', 'idafastamento')
df_limpo = carregar_dimensao(df_limpo, 'UnidadeMedida', 'un_medida', 'un_medida', 'idun_medida')

print("Dimensões básicas carregadas com sucesso!")
print(f"Unidades de medida cadastradas: {df_limpo['idun_medida'].nunique(dropna=False)}")
print(f"Afastamentos cadastrados: {df_limpo['idafastamento'].nunique(dropna=False)}")
print(df_limpo[['Processo', 'Afastamento', 'idafastamento', 'UnidadeMedida', 'idun_medida']].head())


Carregando dimensões fixas (Placeholders)...
Carregando Afastamentos e Unidades de Medida...
Dimensões básicas carregadas com sucesso!
Unidades de medida cadastradas: 83
Afastamentos cadastrados: 7
                   Processo                    Afastamento  idafastamento  \
0  **SEI-120001/009348/2022    Dispensa - Pequenas Compras              1   
1        020002/000001/2024  Inexigibilidade - 14.133/2021              2   
2        020002/000002/2024  Inexigibilidade - 14.133/2021              2   
3        020002/000003/2024  Inexigibilidade - 14.133/2021              2   
4        020002/000004/2024  Inexigibilidade - 14.133/2021              2   

  UnidadeMedida  idun_medida  
0       SERVICO            1  
1       SERVICO            1  
2       SERVICO            1  
3       SERVICO            1  
4       SERVICO            1  


## Carga de Órgãos e Partes (Fornecedores)

In [7]:
# 1. Órgãos (Supertipo)
df_orgaos = df_limpo[['Unidade']].dropna().drop_duplicates().rename(columns={'Unidade': 'nome_orgao'})

# Verifica o que já existe no banco
df_orgao_existente = pd.read_sql("SELECT nome_orgao FROM orgao", engine)
df_orgaos_inserir = df_orgaos[~df_orgaos['nome_orgao'].isin(df_orgao_existente['nome_orgao'])]

if not df_orgaos_inserir.empty:
    df_orgaos_inserir.to_sql('orgao', engine, if_exists='append', index=False)
    print(f"   - {len(df_orgaos_inserir)} novos órgãos inseridos.")

# Mapeia os IDs dos órgãos de volta no df_limpo
df_orgao_db = pd.read_sql("SELECT idorgao, nome_orgao FROM orgao", engine)
df_limpo = df_limpo.merge(df_orgao_db, left_on='Unidade', right_on='nome_orgao', how='left')

# 2. Órgão Estadual (Subtipo)
# Verifica quais órgãos do banco ainda não estão na tabela orgao_estadual
df_orgao_estadual_existente = pd.read_sql("SELECT orgao_idorgao FROM orgao_estadual", engine)
orgaos_para_subtabela = df_orgao_db[~df_orgao_db['idorgao'].isin(df_orgao_estadual_existente['orgao_idorgao'])]

if not orgaos_para_subtabela.empty:
    df_sub_estadual = pd.DataFrame({
        'orgao_idorgao': orgaos_para_subtabela['idorgao'],
        'uf_iduf': uf_id  # Variável criada na célula anterior
    })
    df_sub_estadual.to_sql('orgao_estadual', engine, if_exists='append', index=False)
    print(f"   - {len(df_sub_estadual)} registros inseridos em orgao_estadual.")

# 3. Partes (Supertipo)
df_partes = df_limpo[['FornecedorVencedor']].dropna().drop_duplicates()

# Busca os fornecedores que já existem (fazendo JOIN com a tabela PJ para pegar a Razão Social)
df_parte_existente = pd.read_sql(
    "SELECT p.idparte, pj.razao_social AS nome FROM parte p JOIN parte_pj pj ON p.idparte = pj.parte_idparte",
    engine
)

# Identifica quem são os novos fornecedores
partes_novas = df_partes[~df_partes['FornecedorVencedor'].isin(df_parte_existente['nome'].dropna())]

if not partes_novas.empty:
    # a) Insere na tabela pai (parte)
    df_novas_partes_insert = pd.DataFrame({'tipo_pessoa': 'PJ', 'cep': None}, index=partes_novas.index)
    df_novas_partes_insert.to_sql('parte', engine, if_exists='append', index=False)

    # b) Como o to_sql não retorna os IDs gerados, buscamos os últimos inseridos
    # (Isso funciona porque acabamos de inserir e não há concorrência no banco)
    df_parte_db = pd.read_sql(f"SELECT idparte FROM parte ORDER BY idparte DESC LIMIT {len(partes_novas)}", engine)

    # c) Atribui os IDs recém-criados às novas partes
    partes_novas = partes_novas.copy()
    partes_novas['idparte'] = df_parte_db['idparte'].values

    # d) Insere na tabela filha (parte_pj)
    df_pj_inserir = pd.DataFrame({
        'parte_idparte': partes_novas['idparte'],
        'natureza_juridica_idnatureza_juridica': natjur_id, # Variável da célula anterior
        'razao_social': partes_novas['FornecedorVencedor']
    })
    df_pj_inserir.to_sql('parte_pj', engine, if_exists='append', index=False)
    print(f"   - {len(partes_novas)} novos fornecedores (PJ) inseridos.")

# 4. Atualiza o df_limpo com os IDs de todas as partes (novas e antigas)
df_parte_final = pd.read_sql(
    "SELECT p.idparte, pj.razao_social AS nome FROM parte p JOIN parte_pj pj ON p.idparte = pj.parte_idparte",
    engine
)
df_limpo = df_limpo.merge(df_parte_final, left_on='FornecedorVencedor', right_on='nome', how='left')

# Remove colunas auxiliares que não precisamos mais
df_limpo = df_limpo.drop(columns=['nome_orgao', 'nome'], errors='ignore')

print("Órgãos e Partes carregados e mapeados com sucesso!")
print(df_limpo[['Processo', 'Unidade', 'idorgao', 'FornecedorVencedor', 'idparte']].head())

   - 99 novos órgãos inseridos.
   - 99 registros inseridos em orgao_estadual.
   - 4001 novos fornecedores (PJ) inseridos.
Órgãos e Partes carregados e mapeados com sucesso!
                   Processo  \
0  **SEI-120001/009348/2022   
1        020002/000001/2024   
2        020002/000002/2024   
3        020002/000003/2024   
4        020002/000004/2024   

                                             Unidade  idorgao  \
0  SEPLAG - Secretaria de Estado de Planejamento ...        1   
1  EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...        2   
2  EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...        2   
3  EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...        2   
4  EMATER-RIO - EMPR DE ASSIST TÉCN E EXT RURAL D...        2   

                                  FornecedorVencedor  idparte  
0  MAR E FIRE EQUIP. E PROT. CONTRA INCÊNDIO LTDA-ME     4001  
1                       ÁGUAS DAS AGULHAS NEGRAS S/A     4000  
2  AUTARQUIA MUNICIPAL DE ÁGUA E ESGOTO DE CACHOE...   

## Carga de itens, Processos e Itens do Processo

In [8]:
# Recupera o ID do tipo de contratação

tipo_contratacao_id = pd.read_sql(
    "SELECT idtipo_contratacao FROM tipo_contratacao WHERE descricao = 'Compra Direta'",
    engine
)['idtipo_contratacao'].values[0]

# CARGA DE ITENS (Sem truncamento)

# Extrai únicos diretamente, sem truncar a string
df_itens = df_limpo[['Item', 'idun_medida']].dropna().drop_duplicates(subset=['Item']).rename(columns={'Item': 'descricao_item'})

df_itens_db = pd.DataFrame({
    'descricao_item': df_itens['descricao_item'],
    'un_medida_idun_medida': df_itens['idun_medida'],
    'categoria_item_idcategoria_item': catitem_id
})
df_itens_db.to_sql('item', engine, if_exists='append', index=False)
print(f"   - {len(df_itens_db)} itens inseridos com texto completo.")

# Mapeia IDs dos itens de volta (usando a string original)
df_item_ref = pd.read_sql("SELECT iditem, descricao_item FROM item", engine)
df_limpo = df_limpo.merge(df_item_ref, left_on='Item', right_on='descricao_item', how='left')
df_limpo = df_limpo.drop(columns=['descricao_item'], errors='ignore')

# CARGA DE PROCESSOS

df_processos = df_limpo.groupby('Processo').agg({
    'ValorProcesso': 'max',
    'Objeto': 'first',
    'AnoProcesso': 'max',
    'DataAprovacao': 'max',
    'EnquadramentoLegal': 'first',
    'idorgao': 'first',
    'idparte': 'first',
    'idafastamento': 'first'
}).reset_index()

# Mantém o truncamento no Objeto e Enquadramento para respeitar o VARCHAR(200)
df_processos['Objeto'] = df_processos['Objeto'].astype(str).str.slice(0, 200)
df_processos['EnquadramentoLegal'] = df_processos['EnquadramentoLegal'].astype(str).str.slice(0, 200)

df_proc_db = pd.DataFrame({
    'numero_processo': df_processos['Processo'],
    'valor_processo': df_processos['ValorProcesso'],
    'objeto': df_processos['Objeto'],
    'ano_processo': df_processos['AnoProcesso'],
    'data_aprovacao': df_processos['DataAprovacao'],
    'enquadramento': df_processos['EnquadramentoLegal'],
    'orgao_idorgao': df_processos['idorgao'],
    'parte_idparte': df_processos['idparte'],
    'tipo_contratacao_idtipo_contratacao': tipo_contratacao_id
})
df_proc_db.to_sql('processo', engine, if_exists='append', index=False)
print(f"   - {len(df_proc_db)} processos inseridos.")

df_proc_ref = pd.read_sql("SELECT idprocesso, numero_processo FROM processo", engine)
df_limpo = df_limpo.merge(df_proc_ref, left_on='Processo', right_on='numero_processo', how='left')
df_limpo = df_limpo.drop(columns=['numero_processo'], errors='ignore')

# CARGA DE PROCESSO COMPRA DIRETA

df_pcd_inserir = df_proc_ref.merge(
    df_processos[['Processo', 'idafastamento']],
    left_on='numero_processo', right_on='Processo'
)

df_pcd_db = pd.DataFrame({
    'processo_idprocesso': df_pcd_inserir['idprocesso'],
    'afastamento_idafastamento': df_pcd_inserir['idafastamento']
})
df_pcd_db.to_sql('processo_compra_direta', engine, if_exists='append', index=False)
print(f"   - {len(df_pcd_db)} registros de compra direta inseridos.")

# CARGA DE ITENS DO PROCESSO (FATOS)

df_itens_proc = df_limpo[['idprocesso', 'iditem', 'Quantidade', 'ValorUnitario']].copy()
df_itens_proc.rename(columns={
    'idprocesso': 'processo_idprocesso',
    'iditem': 'item_iditem',
    'Quantidade': 'quantidade',
    'ValorUnitario': 'valor_unitario'
}, inplace=True)

df_itens_proc = df_itens_proc.dropna(subset=['processo_idprocesso', 'item_iditem'])

# Insere todas as linhas
df_itens_proc.to_sql('itens_processo', engine, if_exists='append', index=False)
print(f"   - {len(df_itens_proc)} itens de processo inseridos.")

print("CARGA FINALIZADA COM SUCESSO! O banco está consistente e sem perda semântica de itens.")

   - 7971 itens inseridos com texto completo.
   - 12228 processos inseridos.
   - 12228 registros de compra direta inseridos.
   - 20642 itens de processo inseridos.
CARGA FINALIZADA COM SUCESSO! O banco está consistente e sem perda semântica de itens.
